In [2]:
!pip install polars pyarrow
!pip install ftfy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import polars as pl

# Definir la ruta absoluta de tu archivo
ruta_absoluta = r'C:\Users\ASUS\Documents\Project_ETL_1\archive\df_icfes_historico.csv'

# Cargar la base de datos con Polars
# Nota: Si el archivo usa punto y coma (;), Polars lo detectará automáticamente en la mayoría de los casos.
df = pl.read_csv(ruta_absoluta)

# Mostrar la cantidad de filas y columnas (Formato: (filas, columnas))
print(f"Dimensiones del archivo: {df.shape}")

# Ver las primeras filas en pantalla
print(df.head())

Dimensiones del archivo: (4629768, 94)
shape: (5, 94)
┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ periodo ┆ estu_conse ┆ estu_estu ┆ estu_tipo ┆ … ┆ estu_hora ┆ fami_posi ┆ estu_tiem ┆ estu_desp │
│ ---     ┆ cutivo     ┆ diante    ┆ documento ┆   ┆ strabnore ┆ cionherma ┆ pocasaaco ┆ lazacoleg │
│ i64     ┆ ---        ┆ ---       ┆ ---       ┆   ┆ mu        ┆ nos       ┆ le        ┆ io        │
│         ┆ str        ┆ str       ┆ str       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│         ┆            ┆           ┆           ┆   ┆ str       ┆ str       ┆ str       ┆ str       │
╞═════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 20191   ┆ SB11201910 ┆ ESTUDIANT ┆ CC        ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│         ┆ 036325     ┆ E         ┆           ┆   ┆           ┆           ┆           ┆           │
│ 20191   ┆ SB11201910 ┆ ESTUDIANT ┆ 

##LIMPIEZA

In [4]:
import polars as pl

# Lista de columnas asignadas
cols_pa_limpiar = [
    "periodo",
    "estu_consecutivo",
    "estu_depto_reside",
    "estu_mcpio_reside",
    "cole_codigo_icfes",
    "cole_nombre_establecimiento",
    "cole_naturaleza",
    "cole_calendario",
    "cole_area_ubicacion"
]

# Reporte de nulos y tipos de datos de todas tus columnas en un solo paso
print("LOS NULOS")
print(df.select(cols_pa_limpiar).null_count())

print("\n VER TIPO DE DATOS")
print(df.select(cols_pa_limpiar).schema)

LOS NULOS
shape: (1, 9)
┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ periodo ┆ estu_conse ┆ estu_dept ┆ estu_mcpi ┆ … ┆ cole_nomb ┆ cole_natu ┆ cole_cale ┆ cole_area │
│ ---     ┆ cutivo     ┆ o_reside  ┆ o_reside  ┆   ┆ re_establ ┆ raleza    ┆ ndario    ┆ _ubicacio │
│ u32     ┆ ---        ┆ ---       ┆ ---       ┆   ┆ ecimiento ┆ ---       ┆ ---       ┆ n         │
│         ┆ u32        ┆ u32       ┆ u32       ┆   ┆ ---       ┆ u32       ┆ u32       ┆ ---       │
│         ┆            ┆           ┆           ┆   ┆ u32       ┆           ┆           ┆ u32       │
╞═════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0       ┆ 0          ┆ 8393      ┆ 8393      ┆ … ┆ 672693    ┆ 672590    ┆ 672693    ┆ 672590    │
└─────────┴────────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘

 VER TIPO DE DATOS
Schema({'periodo': Int64, 'estu_consecutivo': S

In [5]:
# Limpiez# de llaves de negocio
df = df.with_columns(
    estu_consecutivo = pl.col("estu_consecutivo").cast(pl.String).str.strip_chars(),
    cole_codigo_icfes = pl.col("cole_codigo_icfes").cast(pl.String).str.strip_chars()
)

# Muestra los tipos de datos y los 5 primeros valores
print(df.select(["estu_consecutivo", "cole_codigo_icfes"]).schema)
print(df.select(["estu_consecutivo", "cole_codigo_icfes"]).head(5))


Schema({'estu_consecutivo': String, 'cole_codigo_icfes': String})
shape: (5, 2)
┌──────────────────┬───────────────────┐
│ estu_consecutivo ┆ cole_codigo_icfes │
│ ---              ┆ ---               │
│ str              ┆ str               │
╞══════════════════╪═══════════════════╡
│ SB11201910036325 ┆ 113357.0          │
│ SB11201910022740 ┆ 101048.0          │
│ SB11201910029970 ┆ 127944.0          │
│ SB11201910030392 ┆ 129650.0          │
│ SB11201910027976 ┆ 74732.0           │
└──────────────────┴───────────────────┘


In [6]:
# Formateo numérico y manejo de nulos específicos para cole_codigo_icfes
df = df.with_columns(
    cole_codigo_icfes = (
        pl.col("cole_codigo_icfes")
        .str.replace(r"\.0$", "")
        .fill_null("SIN INFORMACION")
        .replace("", "SIN INFORMACION")
    )
)

print(df.select("cole_codigo_icfes").head(5))


shape: (5, 1)
┌───────────────────┐
│ cole_codigo_icfes │
│ ---               │
│ str               │
╞═══════════════════╡
│ 113357            │
│ 101048            │
│ 127944            │
│ 129650            │
│ 74732             │
└───────────────────┘


In [7]:
# Estandarización de periodo y creación del año
df = df.with_columns(
    periodo = pl.col("periodo").cast(pl.String).str.strip_chars(),
    anio = pl.col("periodo").cast(pl.String).str.strip_chars().str.slice(0, 4).cast(pl.Int32)
)
# Muestra las combinaciones únicas creadas de periodo y año
print(df.select(["periodo", "anio"]).unique().sort("periodo"))

shape: (14, 2)
┌─────────┬──────┐
│ periodo ┆ anio │
│ ---     ┆ ---  │
│ str     ┆ i32  │
╞═════════╪══════╡
│ 20191   ┆ 2019 │
│ 20192   ┆ 2019 │
│ 20201   ┆ 2020 │
│ 20202   ┆ 2020 │
│ 20211   ┆ 2021 │
│ …       ┆ …    │
│ 20232   ┆ 2023 │
│ 20241   ┆ 2024 │
│ 20242   ┆ 2024 │
│ 20251   ┆ 2025 │
│ 20252   ┆ 2025 │
└─────────┴──────┘


In [8]:
import ftfy

# 1. Crear diccionarios evaluando ftfy sobre los valores ya estandarizados

dict_deptos = { 
    val: ftfy.fix_text(val) 
    for val in df["estu_depto_reside"].unique() 
    if val not in [None, "SIN INFORMACION"] 
}

dict_mcpios = { 
    val: ftfy.fix_text(val) 
    for val in df["estu_mcpio_reside"].unique() 
    if val not in [None, "SIN INFORMACION"] 
}

# 2. Reemplazo vectorial nativo en Polars 
df = df.with_columns(
    estu_depto_reside = pl.col("estu_depto_reside").replace(dict_deptos),
    estu_mcpio_reside = pl.col("estu_mcpio_reside").replace(dict_mcpios)
)

with pl.Config(tbl_rows=50):
    print(df.select(["estu_depto_reside", "estu_mcpio_reside"]).head(50))


shape: (50, 2)
┌───────────────────┬─────────────────────┐
│ estu_depto_reside ┆ estu_mcpio_reside   │
│ ---               ┆ ---                 │
│ str               ┆ str                 │
╞═══════════════════╪═════════════════════╡
│ ANTIOQUIA         ┆ MEDELLÍN            │
│ VALLE             ┆ PALMIRA             │
│ META              ┆ VILLAVICENCIO       │
│ VALLE             ┆ CALI                │
│ VALLE             ┆ CAICEDONIA          │
│ BOGOTÁ            ┆ BOGOTÁ D.C.         │
│ CUNDINAMARCA      ┆ ZIPAQUIRÁ           │
│ VALLE             ┆ EL CERRITO          │
│ NARIÑO            ┆ PASTO               │
│ TOLIMA            ┆ IBAGUÉ              │
│ null              ┆ null                │
│ NORTE SANTANDER   ┆ CÚCUTA              │
│ ANTIOQUIA         ┆ ENVIGADO            │
│ SANTANDER         ┆ BUCARAMANGA         │
│ CUNDINAMARCA      ┆ FUNZA               │
│ BOGOTÁ            ┆ BOGOTÁ D.C.         │
│ NARIÑO            ┆ PASTO               │
│ BOGOTÁ         

In [ ]:
# Estandarización geográfica y tratamiento de nulos/vacíos
df = df.with_columns(
    estu_depto_reside = pl.col("estu_depto_reside").cast(pl.String).str.strip_chars().str.to_uppercase().fill_null("SIN INFORMACION").replace("", "SIN INFORMACION"),
    estu_mcpio_reside = pl.col("estu_mcpio_reside").cast(pl.String).str.strip_chars().str.to_uppercase().fill_null("SIN INFORMACION").replace("", "SIN INFORMACION")
)

print(df.select(["estu_depto_reside", "estu_mcpio_reside"]).null_count())
print(df.select(["estu_depto_reside", "estu_mcpio_reside"]).head(50))


shape: (1, 2)
┌───────────────────┬───────────────────┐
│ estu_depto_reside ┆ estu_mcpio_reside │
│ ---               ┆ ---               │
│ u32               ┆ u32               │
╞═══════════════════╪═══════════════════╡
│ 0                 ┆ 0                 │
└───────────────────┴───────────────────┘
shape: (10, 2)
┌───────────────────┬───────────────────┐
│ estu_depto_reside ┆ estu_mcpio_reside │
│ ---               ┆ ---               │
│ str               ┆ str               │
╞═══════════════════╪═══════════════════╡
│ ANTIOQUIA         ┆ MEDELLÍN          │
│ VALLE             ┆ PALMIRA           │
│ META              ┆ VILLAVICENCIO     │
│ VALLE             ┆ CALI              │
│ VALLE             ┆ CAICEDONIA        │
│ BOGOTÁ            ┆ BOGOTÁ D.C.       │
│ CUNDINAMARCA      ┆ ZIPAQUIRÁ         │
│ VALLE             ┆ EL CERRITO        │
│ NARIÑO            ┆ PASTO             │
│ TOLIMA            ┆ IBAGUÉ            │
└───────────────────┴───────────────────┘


In [16]:
# Estandarización de atributos del colegio
df = df.with_columns(
    cole_nombre_establecimiento = pl.col("cole_nombre_establecimiento").cast(pl.String).str.strip_chars().str.to_uppercase().fill_null("SIN INFORMACION").replace("", "SIN INFORMACION"),
    cole_naturaleza = pl.col("cole_naturaleza").cast(pl.String).str.strip_chars().str.to_uppercase().fill_null("SIN INFORMACION").replace("", "SIN INFORMACION"),
    cole_calendario = pl.col("cole_calendario").cast(pl.String).str.strip_chars().str.to_uppercase().fill_null("SIN INFORMACION").replace("", "SIN INFORMACION"),
    cole_area_ubicacion = pl.col("cole_area_ubicacion").cast(pl.String).str.strip_chars().str.to_uppercase().replace({"URBANO": "URBANA"}).fill_null("SIN INFORMACION").replace("", "SIN INFORMACION")
)
print("Naturaleza:", df.get_column("cole_naturaleza").unique().to_list())
print("Calendario:", df.get_column("cole_calendario").unique().to_list())
print("Área Ubicación:", df.get_column("cole_area_ubicacion").unique().to_list())

Naturaleza: ['NO OFICIAL', 'SIN INFORMACION', 'OFICIAL']
Calendario: ['OTRO', 'B', 'A', 'SIN INFORMACION']
Área Ubicación: ['RURAL', 'SIN INFORMACION', 'URBANA']


In [17]:
# Control de duplicados sobre la llave primaria
total_filas = df.height
filas_unicas = df.select(pl.col("estu_consecutivo").n_unique()).item()
total_duplicados = total_filas - filas_unicas

print(f"Total de registros: {total_filas:,}")
print(f"Estudiantes únicos (estu_consecutivo): {filas_unicas:,}")
print(f"Cantidad de registros duplicados: {total_duplicados:,}")

Total de registros: 4,629,766
Estudiantes únicos (estu_consecutivo): 4,629,766
Cantidad de registros duplicados: 0


In [18]:
cols_mostrar = [
    "periodo",
    "anio",
    "estu_consecutivo",
    "estu_depto_reside",
    "estu_mcpio_reside",
    "cole_codigo_icfes",
    "cole_nombre_establecimiento",
    "cole_naturaleza",
    "cole_calendario",
    "cole_area_ubicacion"
]

# Configura Polars para mostrar todas las columnas sin truncarlas en pantalla
with pl.Config(tbl_cols=len(cols_mostrar)):
    duplicados = df.filter(pl.col("estu_consecutivo").is_duplicated()).select(cols_mostrar)
    print(duplicados)

shape: (0, 10)
┌─────────┬──────┬──────────┬──────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│ periodo ┆ anio ┆ estu_con ┆ estu_dep ┆ estu_mc ┆ cole_co ┆ cole_no ┆ cole_na ┆ cole_ca ┆ cole_ar │
│ ---     ┆ ---  ┆ secutivo ┆ to_resid ┆ pio_res ┆ digo_ic ┆ mbre_es ┆ turalez ┆ lendari ┆ ea_ubic │
│ str     ┆ i32  ┆ ---      ┆ e        ┆ ide     ┆ fes     ┆ tableci ┆ a       ┆ o       ┆ acion   │
│         ┆      ┆ str      ┆ ---      ┆ ---     ┆ ---     ┆ miento  ┆ ---     ┆ ---     ┆ ---     │
│         ┆      ┆          ┆ str      ┆ str     ┆ str     ┆ ---     ┆ str     ┆ str     ┆ str     │
│         ┆      ┆          ┆          ┆         ┆         ┆ str     ┆         ┆         ┆         │
╞═════════╪══════╪══════════╪══════════╪═════════╪═════════╪═════════╪═════════╪═════════╪═════════╡
└─────────┴──────┴──────────┴──────────┴─────────┴─────────┴─────────┴─────────┴─────────┴─────────┘


In [ ]:
# Eliminar registros duplicados manteniendo la primera aparición
df = df.unique(subset=["estu_consecutivo"], keep="first")

# Confirmar la limpieza
print(f"Nuevo total de registros: {df.height:,}")
print(f"Estudiantes únicos: {df.select(pl.col('estu_consecutivo').n_unique()).item():,}")

In [13]:

# Lista de columnas asignadas
cols_pa_limpiar = [
    "periodo",
    "anio",
    "estu_consecutivo",
    "estu_depto_reside",
    "estu_mcpio_reside",
    "cole_codigo_icfes",
    "cole_nombre_establecimiento",
    "cole_naturaleza",
    "cole_calendario",
    "cole_area_ubicacion"
]

# Reporte de nulos y tipos de datos de todas tus columnas en un solo paso
print("LOS NULOS")
print(df.select(cols_pa_limpiar).null_count())

print("\n VER TIPO DE DATOS")
print(df.select(cols_pa_limpiar).schema)

LOS NULOS
shape: (1, 10)
┌─────────┬──────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ periodo ┆ anio ┆ estu_conse ┆ estu_depto ┆ … ┆ cole_nombr ┆ cole_natur ┆ cole_calen ┆ cole_area_ │
│ ---     ┆ ---  ┆ cutivo     ┆ _reside    ┆   ┆ e_establec ┆ aleza      ┆ dario      ┆ ubicacion  │
│ u32     ┆ u32  ┆ ---        ┆ ---        ┆   ┆ imiento    ┆ ---        ┆ ---        ┆ ---        │
│         ┆      ┆ u32        ┆ u32        ┆   ┆ ---        ┆ u32        ┆ u32        ┆ u32        │
│         ┆      ┆            ┆            ┆   ┆ u32        ┆            ┆            ┆            │
╞═════════╪══════╪════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 0       ┆ 0    ┆ 0          ┆ 0          ┆ … ┆ 0          ┆ 0          ┆ 0          ┆ 0          │
└─────────┴──────┴────────────┴────────────┴───┴────────────┴────────────┴────────────┴────────────┘

 VER TIPO DE DATOS
Schema({'periodo': String, 'anio': Int32, 'est